In [1]:
from datasets import load_dataset

# Carga el dataset de Hugging Face
dataset = load_dataset("SINAI/ALIA-es-discriminative-stance-detection")

# Inspeccionar las particiones (train, validation, test)
print(dataset)

# Convertir a Pandas DataFrame si prefieres trabajar con tablas tradicionales
train_df = dataset['train'].to_pandas()
print(train_df.head())

/Users/paulaho/opt/anaconda3/envs/tp3_nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 3000/3000 [00:00<00:00, 42373.98 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'target', 'description', 'comment', 'annotation_1', 'annotation_2', 'annotation_3', 'majority_label'],
        num_rows: 3000
    })
})
  id                                             target  \
0  1                   "Bajar" las motos de las aceras.   
1  2                   "Bajar" las motos de las aceras.   
2  3                   "Bajar" las motos de las aceras.   
3  4                   "Bajar" las motos de las aceras.   
4  5  ARREGLAR LOS REBAJES DE LOS BORDILLOS PARA FAC...   

                                         description  \
0  Esta propuesta es originaria del vecino. [NOMB...   
1  Esta propuesta es originaria del vecino. [NOMB...   
2  Los motoristas, aparte del ruido a veces intol...   
3  Los motoristas, aparte del ruido a veces intol...   
4  El mal estado de las calles y aceras ha hecho ...   

                                             comment annotation_1  \
0  A mi no me molesta tanto que aparquen en

In [2]:
import pandas as pd

# Convertimos el dataset a DataFrame
df = dataset["train"].to_pandas()

# Cantidad de comentarios por clase
print(df["majority_label"].value_counts())

majority_label
favor      1780
against     627
neutral     443
            150
Name: count, dtype: int64


In [3]:
# Comentarios sin etiqueta mayoritaria
sin_etiqueta = df[df["majority_label"] == ""]

print("Cantidad:", len(sin_etiqueta))

# Ver las anotaciones de los primeros 10
print(sin_etiqueta[
    ["annotation_1", "annotation_2", "annotation_3"]
].head(10))

Cantidad: 150
    annotation_1 annotation_2 annotation_3
2        against      neutral        favor
27       neutral        favor      against
29         favor      neutral      against
45       against        favor      neutral
58       against      neutral        favor
102        favor      against      neutral
111        favor      neutral      against
154      neutral      against        favor
161      against        favor      neutral
166        favor      against      neutral


In [4]:
# Verificar si los 150 casos tienen tres anotaciones distintas
sin_etiqueta[
    ["annotation_1", "annotation_2", "annotation_3"]
].nunique(axis=1).value_counts()

3    150
Name: count, dtype: int64

In [5]:
# Excluir comentarios sin etiqueta mayoritaria
df_limpio = df[df["majority_label"] != ""].copy()

# Verificar cantidad de comentarios
print("Comentarios originales:", len(df))
print("Comentarios utilizables:", len(df_limpio))

# Ver distribución de clases
print(df_limpio["majority_label"].value_counts())

Comentarios originales: 3000
Comentarios utilizables: 2850
majority_label
favor      1780
against     627
neutral     443
Name: count, dtype: int64


In [6]:
# Contar cuántas etiquetas distintas hay por comentario
acuerdos = df[
    ["annotation_1", "annotation_2", "annotation_3"]
].nunique(axis=1)

# Calcular tipos de acuerdo
acuerdo_total = (acuerdos == 1).sum()
acuerdo_parcial = (acuerdos == 2).sum()
sin_acuerdo = (acuerdos == 3).sum()

print("Acuerdo total:", acuerdo_total)
print("Acuerdo parcial:", acuerdo_parcial)
print("Sin acuerdo:", sin_acuerdo)

# Porcentajes
print("\nPorcentajes:")
print("Acuerdo total:", round(acuerdo_total / len(df) * 100, 2), "%")
print("Acuerdo parcial:", round(acuerdo_parcial / len(df) * 100, 2), "%")
print("Sin acuerdo:", round(sin_acuerdo / len(df) * 100, 2), "%")

Acuerdo total: 1539
Acuerdo parcial: 1311
Sin acuerdo: 150

Porcentajes:
Acuerdo total: 51.3 %
Acuerdo parcial: 43.7 %
Sin acuerdo: 5.0 %


In [7]:
# Calcular cantidad de palabras por comentario
df_limpio["num_palabras"] = df_limpio["comment"].str.split().str.len()

# Mostrar estadísticas
print(df_limpio["num_palabras"].describe())

count    2850.000000
mean       54.138596
std        32.073506
min         7.000000
25%        28.000000
50%        46.000000
75%        76.000000
max       155.000000
Name: num_palabras, dtype: float64
